# Mangrove avoided EAD and Hurricane Melissa damage

This notebook joins patch-level mangrove coastal-flood avoided EAD attribution to Hurricane Melissa NDVI damage metrics. It reports how much of the mangrove area associated with positive avoided EAD was damaged, then summarises the results by distance from the NOAA best-track line and by NOAA wind-speed swath threshold.

Damage definition follows the existing Hurricane Melissa mangrove notebooks: a pixel is damaged where `(after2 - before) / before < -0.10`, evaluated only where pre-event NDVI is at least `0.20`.

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from shapely.ops import unary_union
from IPython.display import display

pd.set_option("display.max_columns", 200)


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "dphil_papers" / "dphil_paper_3").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing dphil_papers/dphil_paper_3")


ROOT = find_project_root()
PAPER3 = ROOT / "dphil_papers" / "dphil_paper_3"
JAMAICA_METRIC_CRS = "EPSG:3448"

REL_DAMAGE_THRESHOLD = -0.10
REL_BASELINE_MIN = 0.20
DISTANCE_BINS_KM = [0, 25, 50, 75, 100, 150, 200, 300]

mangroves_path = PAPER3 / "inputs/forces_of_nature_mangroves/mangroves.shp"
damage_by_id_path = PAPER3 / "results/threats/ndvi/mangrove_avoided_ead_damage_recovery/mangrove_ndvi_damage_recovery_by_id.csv"
ndvi_before_path = PAPER3 / "inputs/ndvi/HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif"

ead_attribution_paths = {
    "minimum": PAPER3 / "results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario/damage_estimates/mangrove_attribution_area_distance_all_sectors_signed/mangrove_attribution_total_all_sectors_signed_area_distance_5000m_nn_fallback.gpkg",
    "maximum": PAPER3 / "results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario/damage_estimates/mangrove_attribution_area_distance_all_sectors_signed/mangrove_attribution_total_all_sectors_signed_area_distance_5000m_nn_fallback.gpkg",
}

track_dir = PAPER3 / "inputs/hurricane_melissa_track_noaa/al132025_best_track"
track_line_path = track_dir / "AL132025_lin.shp"
windswath_path = track_dir / "AL132025_windswath.shp"

output_dir = PAPER3 / "results/threats/hurricane_melissa_damage/mangrove_eads_hurricane_damage"
output_dir.mkdir(parents=True, exist_ok=True)

for path in [mangroves_path, damage_by_id_path, ndvi_before_path, track_line_path, windswath_path, *ead_attribution_paths.values()]:
    if not path.exists():
        raise FileNotFoundError(path)

print("ROOT:", ROOT)
print("Output directory:", output_dir)


In [ ]:
def safe_divide(numerator, denominator, multiplier=1.0):
    return float(numerator / denominator * multiplier) if denominator and denominator != 0 else 0.0


def coerce_track_to_wgs84(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    bounds = gdf.total_bounds
    looks_like_lonlat = (
        abs(bounds[0]) <= 180
        and abs(bounds[2]) <= 180
        and abs(bounds[1]) <= 90
        and abs(bounds[3]) <= 90
    )
    if looks_like_lonlat:
        return gdf.set_crs(4326, allow_override=True)
    if gdf.crs is None:
        return gdf.set_crs(4326, allow_override=True)
    return gdf.to_crs(4326)


def load_scenario_ead(path: Path, scenario: str) -> pd.DataFrame:
    gdf = gpd.read_file(path)
    df = pd.DataFrame(gdf.drop(columns="geometry"))
    df["Mangrove_ID"] = pd.to_numeric(df["Mangrove_ID"], errors="coerce").astype("Int64")
    suffix = "min" if scenario == "minimum" else "max"
    return df[
        [
            "Mangrove_ID",
            "Positive_Avoided_EAD_USD_attributed",
            "Net_Avoided_EAD_USD_attributed",
            "Net_Impact_Class",
        ]
    ].rename(
        columns={
            "Positive_Avoided_EAD_USD_attributed": f"positive_avoided_ead_usd_{suffix}",
            "Net_Avoided_EAD_USD_attributed": f"net_ead_usd_{suffix}",
            "Net_Impact_Class": f"net_impact_class_{suffix}",
        }
    )


def summary_record(df: pd.DataFrame) -> dict:
    damaged = df[df["has_gt10pct_damage_any_pixel"]].copy()

    total_area_ha = df["area_ha"].sum()
    eligible_area_ha = df["eligible_area_ha"].sum()
    damaged_area_ha = df["damaged_area_ha"].sum()
    ead_min = df["positive_avoided_ead_usd_min"].sum()
    ead_max = df["positive_avoided_ead_usd_max"].sum()
    damaged_ead_min = damaged["positive_avoided_ead_usd_min"].sum()
    damaged_ead_max = damaged["positive_avoided_ead_usd_max"].sum()

    return {
        "patch_count": int(len(df)),
        "mangrove_area_ha": float(total_area_ha),
        "eligible_baseline_area_ha": float(eligible_area_ha),
        "damaged_gt10pct_area_ha": float(damaged_area_ha),
        "pct_eligible_area_damaged_gt10pct": safe_divide(damaged_area_ha, eligible_area_ha, 100),
        "patches_with_any_gt10pct_damage": int(len(damaged)),
        "pct_patches_with_any_gt10pct_damage": safe_divide(len(damaged), len(df), 100),
        "mangrove_area_ha_with_any_gt10pct_damage": float(damaged["area_ha"].sum()),
        "pct_mangrove_area_with_any_gt10pct_damage": safe_divide(damaged["area_ha"].sum(), total_area_ha, 100),
        "positive_avoided_ead_usd_min": float(ead_min),
        "positive_avoided_ead_usd_max": float(ead_max),
        "positive_avoided_ead_usd_min_any_damage_patch": float(damaged_ead_min),
        "positive_avoided_ead_usd_max_any_damage_patch": float(damaged_ead_max),
        "pct_positive_avoided_ead_min_any_damage_patch": safe_divide(damaged_ead_min, ead_min, 100),
        "pct_positive_avoided_ead_max_any_damage_patch": safe_divide(damaged_ead_max, ead_max, 100),
        "mean_pct_damaged_gt10pct_of_eligible_unweighted": float(df["pct_damaged_gt10pct_after2_of_eligible"].fillna(0).mean()) if len(df) else 0.0,
        "mean_pct_damaged_gt10pct_of_eligible_weighted_by_eligible_pixels": float(np.average(
            df["pct_damaged_gt10pct_after2_of_eligible"].fillna(0),
            weights=df["n_eligible_baseline_ge_0_2"].fillna(0),
        )) if df["n_eligible_baseline_ge_0_2"].fillna(0).sum() > 0 else 0.0,
    }


def scenario_summary_record(df: pd.DataFrame, scenario: str, ead_col: str, net_col: str) -> dict:
    scenario_df = df[df[ead_col] > 0].copy()
    damaged = scenario_df[scenario_df["has_gt10pct_damage_any_pixel"]].copy()
    total_ead = scenario_df[ead_col].sum()
    damaged_ead = damaged[ead_col].sum()
    return {
        "scenario": scenario,
        "patch_count": int(len(scenario_df)),
        "mangrove_area_ha": float(scenario_df["area_ha"].sum()),
        "eligible_baseline_area_ha": float(scenario_df["eligible_area_ha"].sum()),
        "damaged_gt10pct_area_ha": float(scenario_df["damaged_area_ha"].sum()),
        "pct_eligible_area_damaged_gt10pct": safe_divide(scenario_df["damaged_area_ha"].sum(), scenario_df["eligible_area_ha"].sum(), 100),
        "patches_with_any_gt10pct_damage": int(len(damaged)),
        "pct_patches_with_any_gt10pct_damage": safe_divide(len(damaged), len(scenario_df), 100),
        "mangrove_area_ha_with_any_gt10pct_damage": float(damaged["area_ha"].sum()),
        "pct_mangrove_area_with_any_gt10pct_damage": safe_divide(damaged["area_ha"].sum(), scenario_df["area_ha"].sum(), 100),
        "positive_avoided_ead_usd": float(total_ead),
        "positive_avoided_ead_usd_any_damage_patch": float(damaged_ead),
        "pct_positive_avoided_ead_any_damage_patch": safe_divide(damaged_ead, total_ead, 100),
        "net_ead_usd": float(scenario_df[net_col].sum()),
    }


In [ ]:
# Load existing Hurricane Melissa NDVI damage metrics by mangrove ID.
damage = pd.read_csv(damage_by_id_path)
damage["Mangrove_ID"] = pd.to_numeric(damage["Mangrove_ID"], errors="coerce").astype("Int64")

with rasterio.open(ndvi_before_path) as src:
    pixel_area_ha = abs(src.transform.a * src.transform.e) / 10_000

damage["eligible_area_ha"] = damage["n_eligible_baseline_ge_0_2"].fillna(0) * pixel_area_ha
damage["damaged_area_ha"] = damage["n_damaged_gt10pct_after2"].fillna(0) * pixel_area_ha
damage["has_gt10pct_damage_any_pixel"] = damage["n_damaged_gt10pct_after2"].fillna(0) > 0
damage["has_gt10pct_damage_ge10pct_of_eligible"] = damage["pct_damaged_gt10pct_after2_of_eligible"].fillna(0) >= 10

# Load current minimum/maximum weighted area-distance mangrove EAD attribution.
ead_min = load_scenario_ead(ead_attribution_paths["minimum"], "minimum")
ead_max = load_scenario_ead(ead_attribution_paths["maximum"], "maximum")

# Load geometries for distance-to-track and wind-swath exposure metrics.
mangroves = gpd.read_file(mangroves_path).to_crs(JAMAICA_METRIC_CRS)
mangroves["Mangrove_ID"] = pd.to_numeric(mangroves["ID"], errors="coerce").astype("Int64")

track_line = coerce_track_to_wgs84(gpd.read_file(track_line_path)).to_crs(JAMAICA_METRIC_CRS)
track_line_union = unary_union(track_line.geometry.tolist())

windswath = coerce_track_to_wgs84(gpd.read_file(windswath_path)).to_crs(JAMAICA_METRIC_CRS)
windswath["RADII"] = pd.to_numeric(windswath["RADII"], errors="coerce")
wind_threshold_unions = {
    threshold: unary_union(windswath.loc[windswath["RADII"] >= threshold, "geometry"].tolist())
    for threshold in [34, 50, 64]
}

geom_metrics = mangroves[["Mangrove_ID", "geometry"]].copy()
geom_metrics["distance_to_track_km"] = geom_metrics.geometry.distance(track_line_union) / 1000

for threshold, wind_union in wind_threshold_unions.items():
    geom_metrics[f"intersects_{threshold}kt_swath"] = geom_metrics.geometry.intersects(wind_union)
    geom_metrics[f"area_ha_in_{threshold}kt_swath"] = geom_metrics.geometry.intersection(wind_union).area / 10_000
    geom_metrics[f"pct_area_in_{threshold}kt_swath"] = np.where(
        geom_metrics.geometry.area > 0,
        geom_metrics.geometry.intersection(wind_union).area / geom_metrics.geometry.area * 100,
        0,
    )


def max_wind_threshold(row):
    for threshold in [64, 50, 34]:
        if row[f"intersects_{threshold}kt_swath"]:
            return f">={threshold} kt"
    return "outside 34 kt swath"


geom_metrics["max_wind_threshold_intersected"] = geom_metrics.apply(max_wind_threshold, axis=1)

patch_table = (
    damage
    .merge(ead_min, on="Mangrove_ID", how="left")
    .merge(ead_max, on="Mangrove_ID", how="left")
    .merge(pd.DataFrame(geom_metrics.drop(columns="geometry")), on="Mangrove_ID", how="left")
)

for col in ["positive_avoided_ead_usd_min", "positive_avoided_ead_usd_max", "net_ead_usd_min", "net_ead_usd_max"]:
    patch_table[col] = patch_table[col].fillna(0.0)

patch_table["positive_avoided_ead_either"] = (
    (patch_table["positive_avoided_ead_usd_min"] > 0)
    | (patch_table["positive_avoided_ead_usd_max"] > 0)
)
patch_table["distance_bin_km"] = pd.cut(
    patch_table["distance_to_track_km"],
    bins=DISTANCE_BINS_KM,
    include_lowest=True,
    right=False,
)

patch_out = output_dir / "mangrove_ead_hurricane_damage_patch_table.csv"
patch_table.to_csv(patch_out, index=False)

print(f"Pixel area: {pixel_area_ha:.4f} ha")
print(f"Saved: {patch_out}")
display(patch_table.head())


In [ ]:
benefit_mangroves = patch_table[patch_table["positive_avoided_ead_either"]].copy()

overall_summary = pd.DataFrame([
    {"analysis_group": "positive_avoided_ead_either", **summary_record(benefit_mangroves)}
])

scenario_summary = pd.DataFrame([
    scenario_summary_record(
        patch_table,
        scenario="minimum",
        ead_col="positive_avoided_ead_usd_min",
        net_col="net_ead_usd_min",
    ),
    scenario_summary_record(
        patch_table,
        scenario="maximum",
        ead_col="positive_avoided_ead_usd_max",
        net_col="net_ead_usd_max",
    ),
])

overall_out = output_dir / "mangrove_ead_hurricane_damage_overall_summary.csv"
scenario_out = output_dir / "mangrove_ead_hurricane_damage_scenario_summary.csv"
overall_summary.to_csv(overall_out, index=False)
scenario_summary.to_csv(scenario_out, index=False)

display(overall_summary)
display(scenario_summary)

print(f"Saved: {overall_out}")
print(f"Saved: {scenario_out}")


In [ ]:
distance_rows = []
for distance_bin in benefit_mangroves["distance_bin_km"].cat.categories:
    subset = benefit_mangroves[benefit_mangroves["distance_bin_km"] == distance_bin]
    distance_rows.append({"distance_bin_km": str(distance_bin), **summary_record(subset)})

distance_summary = pd.DataFrame(distance_rows)

wind_threshold_rows = []
for threshold in [34, 50, 64]:
    subset = benefit_mangroves[benefit_mangroves[f"intersects_{threshold}kt_swath"]]
    wind_threshold_rows.append({"wind_threshold": f">={threshold} kt", **summary_record(subset)})

wind_threshold_summary = pd.DataFrame(wind_threshold_rows)

wind_zone_rows = []
for wind_zone in [">=64 kt", ">=50 kt", ">=34 kt", "outside 34 kt swath"]:
    subset = benefit_mangroves[benefit_mangroves["max_wind_threshold_intersected"] == wind_zone]
    if len(subset) == 0:
        continue
    wind_zone_rows.append({"wind_zone_exclusive": wind_zone, **summary_record(subset)})

wind_zone_summary = pd.DataFrame(wind_zone_rows)

distance_out = output_dir / "mangrove_ead_hurricane_damage_by_distance_bin.csv"
wind_threshold_out = output_dir / "mangrove_ead_hurricane_damage_by_wind_threshold_cumulative.csv"
wind_zone_out = output_dir / "mangrove_ead_hurricane_damage_by_wind_zone_exclusive.csv"

distance_summary.to_csv(distance_out, index=False)
wind_threshold_summary.to_csv(wind_threshold_out, index=False)
wind_zone_summary.to_csv(wind_zone_out, index=False)

display(distance_summary)
display(wind_threshold_summary)
display(wind_zone_summary)

print(f"Saved: {distance_out}")
print(f"Saved: {wind_threshold_out}")
print(f"Saved: {wind_zone_out}")


## Damage extent and minimum-maximum EAD range

This is the primary interpretation table for comparing hurricane damage with coastal-flood protection value. It classifies each positive avoided-EAD mangrove patch by the percentage of eligible baseline area with >10% NDVI decline, then reports the minimum-maximum positive avoided-EAD range associated with each damage class.

In [ ]:
def assign_damage_extent_class(pct_damaged):
    pct_damaged = 0 if pd.isna(pct_damaged) else float(pct_damaged)
    if pct_damaged <= 0:
        return "No detected >10% damage"
    if pct_damaged < 10:
        return "Low damage (>0-10% eligible area)"
    if pct_damaged < 50:
        return "Moderate damage (10-50% eligible area)"
    return "High damage (>=50% eligible area)"


damage_extent_order = [
    "No detected >10% damage",
    "Low damage (>0-10% eligible area)",
    "Moderate damage (10-50% eligible area)",
    "High damage (>=50% eligible area)",
]

benefit_mangroves = benefit_mangroves.copy()
benefit_mangroves["damage_extent_class"] = benefit_mangroves[
    "pct_damaged_gt10pct_after2_of_eligible"
].apply(assign_damage_extent_class)

damage_extent_rows = []
for damage_extent_class in damage_extent_order:
    subset = benefit_mangroves[benefit_mangroves["damage_extent_class"] == damage_extent_class]
    damage_extent_rows.append(
        {
            "damage_extent_class": damage_extent_class,
            "patch_count": len(subset),
            "mangrove_area_ha": subset["area_ha"].sum(),
            "eligible_baseline_area_ha": subset["eligible_area_ha"].sum(),
            "damaged_gt10pct_area_ha": subset["damaged_area_ha"].sum(),
            "pct_eligible_area_damaged_gt10pct": safe_divide(
                subset["damaged_area_ha"].sum(),
                subset["eligible_area_ha"].sum(),
                100,
            ),
            "positive_avoided_ead_usd_min": subset["positive_avoided_ead_usd_min"].sum(),
            "positive_avoided_ead_usd_max": subset["positive_avoided_ead_usd_max"].sum(),
            "pct_positive_avoided_ead_min": safe_divide(
                subset["positive_avoided_ead_usd_min"].sum(),
                benefit_mangroves["positive_avoided_ead_usd_min"].sum(),
                100,
            ),
            "pct_positive_avoided_ead_max": safe_divide(
                subset["positive_avoided_ead_usd_max"].sum(),
                benefit_mangroves["positive_avoided_ead_usd_max"].sum(),
                100,
            ),
        }
    )

damage_extent_summary = pd.DataFrame(damage_extent_rows)

damage_threshold_rows = []
damage_thresholds = [
    ("Any >10% damaged pixel", benefit_mangroves["pct_damaged_gt10pct_after2_of_eligible"] > 0),
    (">=10% eligible area damaged", benefit_mangroves["pct_damaged_gt10pct_after2_of_eligible"] >= 10),
    (">=25% eligible area damaged", benefit_mangroves["pct_damaged_gt10pct_after2_of_eligible"] >= 25),
    (">=50% eligible area damaged", benefit_mangroves["pct_damaged_gt10pct_after2_of_eligible"] >= 50),
    (">=75% eligible area damaged", benefit_mangroves["pct_damaged_gt10pct_after2_of_eligible"] >= 75),
]

for threshold_label, threshold_mask in damage_thresholds:
    subset = benefit_mangroves[threshold_mask]
    damage_threshold_rows.append(
        {
            "damage_threshold": threshold_label,
            "patch_count": len(subset),
            "mangrove_area_ha": subset["area_ha"].sum(),
            "eligible_baseline_area_ha": subset["eligible_area_ha"].sum(),
            "damaged_gt10pct_area_ha": subset["damaged_area_ha"].sum(),
            "pct_total_eligible_area_damaged_in_threshold_group": safe_divide(
                subset["damaged_area_ha"].sum(),
                benefit_mangroves["eligible_area_ha"].sum(),
                100,
            ),
            "positive_avoided_ead_usd_min": subset["positive_avoided_ead_usd_min"].sum(),
            "positive_avoided_ead_usd_max": subset["positive_avoided_ead_usd_max"].sum(),
            "pct_positive_avoided_ead_min": safe_divide(
                subset["positive_avoided_ead_usd_min"].sum(),
                benefit_mangroves["positive_avoided_ead_usd_min"].sum(),
                100,
            ),
            "pct_positive_avoided_ead_max": safe_divide(
                subset["positive_avoided_ead_usd_max"].sum(),
                benefit_mangroves["positive_avoided_ead_usd_max"].sum(),
                100,
            ),
        }
    )

damage_threshold_summary = pd.DataFrame(damage_threshold_rows)

damage_extent_out = output_dir / "mangrove_ead_hurricane_damage_by_damage_extent_class.csv"
damage_threshold_out = output_dir / "mangrove_ead_hurricane_damage_by_damage_threshold.csv"
damage_extent_summary.to_csv(damage_extent_out, index=False)
damage_threshold_summary.to_csv(damage_threshold_out, index=False)

display(damage_extent_summary)
display(damage_threshold_summary)

print(f"Saved: {damage_extent_out}")
print(f"Saved: {damage_threshold_out}")


In [ ]:
row = overall_summary.iloc[0]
low_damage = damage_extent_summary.loc[
    damage_extent_summary["damage_extent_class"] == "Low damage (>0-10% eligible area)"
].iloc[0]
moderate_damage = damage_extent_summary.loc[
    damage_extent_summary["damage_extent_class"] == "Moderate damage (10-50% eligible area)"
].iloc[0]
high_damage = damage_extent_summary.loc[
    damage_extent_summary["damage_extent_class"] == "High damage (>=50% eligible area)"
].iloc[0]
substantial_damage = damage_threshold_summary.loc[
    damage_threshold_summary["damage_threshold"] == ">=10% eligible area damaged"
].iloc[0]

print("Key reporting values: minimum-maximum avoided-EAD range")
print("-------------------------------------------------------")
print(f"Positive avoided-EAD mangrove patches: {row['patch_count']:.0f}")
print(f"Associated mangrove area: {row['mangrove_area_ha']:,.1f} ha")
print(
    f"Total positive avoided EAD: "
    f"US${row['positive_avoided_ead_usd_min'] / 1e6:.2f}–"
    f"{row['positive_avoided_ead_usd_max'] / 1e6:.2f} million/yr"
)
print(
    f"Low-damage patches (>0-10% eligible area damaged): "
    f"US${low_damage['positive_avoided_ead_usd_min'] / 1e6:.2f}–"
    f"{low_damage['positive_avoided_ead_usd_max'] / 1e6:.2f} million/yr "
    f"({low_damage['pct_positive_avoided_ead_min']:.1f}–"
    f"{low_damage['pct_positive_avoided_ead_max']:.1f}%)"
)
print(
    f"Moderate-damage patches (10-50% eligible area damaged): "
    f"US${moderate_damage['positive_avoided_ead_usd_min'] / 1e6:.2f}–"
    f"{moderate_damage['positive_avoided_ead_usd_max'] / 1e6:.2f} million/yr "
    f"({moderate_damage['pct_positive_avoided_ead_min']:.1f}–"
    f"{moderate_damage['pct_positive_avoided_ead_max']:.1f}%)"
)
print(
    f"High-damage patches (>=50% eligible area damaged): "
    f"US${high_damage['positive_avoided_ead_usd_min'] / 1e6:.2f}–"
    f"{high_damage['positive_avoided_ead_usd_max'] / 1e6:.2f} million/yr "
    f"({high_damage['pct_positive_avoided_ead_min']:.1f}–"
    f"{high_damage['pct_positive_avoided_ead_max']:.1f}%)"
)
print(
    f"Substantial damage threshold (>=10% eligible area damaged): "
    f"US${substantial_damage['positive_avoided_ead_usd_min'] / 1e6:.2f}–"
    f"{substantial_damage['positive_avoided_ead_usd_max'] / 1e6:.2f} million/yr "
    f"({substantial_damage['pct_positive_avoided_ead_min']:.1f}–"
    f"{substantial_damage['pct_positive_avoided_ead_max']:.1f}%)"
)
